In [0]:
from pyspark.sql.functions import col, lit

orders_bronze = spark.table(
    "workspace.default.orders_bronze"
)

In [0]:
existing_orders = (
    orders_bronze
    .limit(5)
)

In [0]:
new_order = (
    orders_bronze
    .limit(1)
    .withColumn("order_id", lit("SIMULATED_NEW_ORDER"))
)

In [0]:
incremental_batch = (
    existing_orders
    .unionByName(new_order)
)

In [0]:
incremental_batch.show(truncate=False)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.orders_bronze"
)

In [0]:
(
    target.alias("target")
    .merge(
        incremental_batch.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
orders_bronze_updated = spark.table(
    "workspace.default.orders_bronze"
)

print("Total rows:", orders_bronze_updated.count())

In [0]:
orders_bronze_updated.filter(
    col("order_id") == "SIMULATED_NEW_ORDER"
).show()

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.orders_bronze"
)

target.delete(
    col("order_id") == "SIMULATED_NEW_ORDER"
)

In [0]:
print(
    spark.table(
        "workspace.default.orders_bronze"
    ).count()
)